# Complete Whisper Pipeline with Comprehensive Evaluation

**Pipeline:**
1. Load training data (80% split)
2. Train Whisper-small model
3. Load test data (20% split)
4. Comprehensive evaluation with multiple metrics

**Metrics:**
- BLEU Score
- chrF++
- BERTScore F1
- WER (Word Error Rate)
- CER (Character Error Rate)
- Word Overlap



In [1]:
!pip install -q transformers datasets librosa soundfile evaluate sacrebleu accelerate bert-score jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 1.3 MB/s eta 0:00:00


In [2]:


# ============================================
# IMPORTS
# ============================================

import torch
import numpy as np
import librosa
import warnings
import os
import gc
import pickle
import zipfile
import pandas as pd
import jiwer
from datetime import datetime
from tqdm import tqdm
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback
)
from datasets import Dataset
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from sacrebleu.metrics import BLEU, CHRF
from bert_score import score as bert_score

warnings.filterwarnings('ignore')

print("=" * 70)
print("COMPLETE WHISPER PIPELINE WITH EVALUATION")
print("=" * 70)

# ============================================
# PART 1: LOAD TRAINING DATA
# ============================================

print("\n" + "=" * 70)
print("PART 1: LOAD TRAINING DATA")
print("=" * 70)

# UPDATE THESE PATHS
TRAIN_AUDIO_DIR = "/kaggle/input/datasets/azkaanasir/15kdatasetapp-mozilla/15ktranlsateddata/12ktrain_dataset/train/AUDIO"
TRAIN_TXT_DIR = "/kaggle/input/datasets/azkaanasir/15kdatasetapp-mozilla/15ktranlsateddata/12ktrain_dataset/train/TXT"

# print(f"\nTraining data paths:")
# print(f"  Audio: {TRAIN_AUDIO_DIR}")
# print(f"  Text:  {TRAIN_TXT_DIR}")

# # Load training data
# train_dataset = []
# audio_files = sorted([f for f in os.listdir(TRAIN_AUDIO_DIR) if f.endswith('.wav')])

# print(f"\nFound {len(audio_files)} training audio files")
# print("Loading training data...")

# for i, audio_file in enumerate(audio_files):
#     if i % 500 == 0:
#         print(f"  Loading {i}/{len(audio_files)}...")
    
#     base_name = os.path.splitext(audio_file)[0]
#     audio_path = os.path.join(TRAIN_AUDIO_DIR, audio_file)
#     txt_path = os.path.join(TRAIN_TXT_DIR, f"{base_name}.txt")
    
#     if not os.path.exists(txt_path):
#         continue
    
#     audio, sr = librosa.load(audio_path, sr=16000)
    
#     with open(txt_path, 'r', encoding='utf-8') as f:
#         english_text = f.read().strip()
    
#     train_dataset.append({
#         'audio': audio,
#         'text': english_text,
#         'filename': audio_file
#     })

# print(f"\nLoaded {len(train_dataset)} training examples")
audio_files = sorted([f for f in os.listdir(TRAIN_AUDIO_DIR) if f.endswith('.wav')])
print(f"Found {len(audio_files)} audio files")

train_manifest = []
for audio_file in audio_files:
    base = os.path.splitext(audio_file)[0]
    txt_path = os.path.join(TRAIN_TXT_DIR, f"{base}.txt")
    if not os.path.exists(txt_path):
        continue
    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read().strip()
    train_manifest.append({
        'audio_path': os.path.join(TRAIN_AUDIO_DIR, audio_file),
        'text': text
    })
print(f"Manifest built: {len(train_manifest)} pairs")
# # Statistics
# durations = [len(d['audio'])/16000 for d in train_dataset]
# text_lengths = [len(d['text'].split()) for d in train_dataset]

# print(f"\nTraining set statistics:")
# print(f"  Audio duration: {min(durations):.1f}s - {max(durations):.1f}s (avg: {np.mean(durations):.1f}s)")
# print(f"  English words: {min(text_lengths)} - {max(text_lengths)} (avg: {np.mean(text_lengths):.1f})")

# # ============================================


COMPLETE WHISPER PIPELINE WITH EVALUATION

PART 1: LOAD TRAINING DATA
Found 11976 audio files
Manifest built: 11976 pairs


In [3]:
# import torch
# import numpy as np
# import librosa
# import warnings
# import os
# import gc
# import pickle
# import zipfile
# import pandas as pd
# import jiwer
# import string
# from datetime import datetime
# from tqdm import tqdm
# from transformers import (
#     WhisperForConditionalGeneration,
#     WhisperProcessor,
#     Seq2SeqTrainingArguments,
#     Seq2SeqTrainer,
#     TrainerCallback
# )
# from dataclasses import dataclass
# from typing import Any, Dict, List, Union
# import evaluate
# from sacrebleu.metrics import BLEU, CHRF
# from bert_score import score as bert_score

# warnings.filterwarnings('ignore')

# print("=" * 70)
# print("COMPLETE WHISPER PIPELINE WITH EVALUATION")
# print("=" * 70)

# # ============================================
# # DATASET CLASS (lazy loading)
# # ============================================

# class BurushaskiDataset(torch.utils.data.Dataset):
#     def __init__(self, manifest, processor):
#         self.manifest = manifest
#         self.processor = processor

#     def __len__(self):
#         return len(self.manifest)

#     def __getitem__(self, idx):
#         item = self.manifest[idx]
#         audio, _ = librosa.load(item['audio_path'], sr=16000)
#         inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt")
#         labels = self.processor.tokenizer(item['text'], return_tensors="pt").input_ids
#         return {
#             "input_features": inputs.input_features.squeeze(0),
#             "labels": labels.squeeze(0)
#         }

# print("Dataset class defined")

# # ============================================
# # PART 1: BUILD MANIFEST (no audio loaded)
# # ============================================

# print("\n" + "=" * 70)
# print("PART 1: LOAD TRAINING DATA")
# print("=" * 70)

# TRAIN_AUDIO_DIR = "/kaggle/input/datasets/azkaanasir/augmented7kdataset/combined/AUDIO"
# TRAIN_TXT_DIR   = "/kaggle/input/datasets/azkaanasir/augmented7kdataset/combined/TXT"

# print(f"\nTraining data paths:")
# print(f"  Audio: {TRAIN_AUDIO_DIR}")
# print(f"  Text:  {TRAIN_TXT_DIR}")

# audio_files = sorted([f for f in os.listdir(TRAIN_AUDIO_DIR) if f.endswith('.wav')])
# print(f"\nFound {len(audio_files)} training audio files")

# train_manifest = []
# for audio_file in audio_files:
#     base = os.path.splitext(audio_file)[0]
#     txt_path = os.path.join(TRAIN_TXT_DIR, f"{base}.txt")
#     if not os.path.exists(txt_path):
#         continue
#     with open(txt_path, 'r', encoding='utf-8') as f:
#         text = f.read().strip()
#     train_manifest.append({
#         'audio_path': os.path.join(TRAIN_AUDIO_DIR, audio_file),
#         'text': text,
#         'filename': audio_file
#     })

# print(f"Manifest built: {len(train_manifest)} pairs (no audio loaded)")

In [4]:
#  # PREPARE DATA FOR WHISPER
# # ============================================

# print("\n" + "=" * 70)
# print("PART 2: PREPARE DATA FOR WHISPER")
# print("=" * 70)

# processor = WhisperProcessor.from_pretrained("openai/whisper-small")
# print("Processor loaded")

# hf_dataset = Dataset.from_dict({
#     "audio": [d["audio"] for d in train_dataset],
#     "text": [d["text"] for d in train_dataset]
# })

# model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
# model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="translate")

# def prepare_dataset(batch):
#     batch["input_features"] = processor(
#         batch["audio"],
#         sampling_rate=16000,
#         return_tensors=None
#     ).input_features[0]
    
#     batch["labels"] = processor(text=batch["text"], return_tensors=None).input_ids
#     return batch

# print("\nProcessing dataset...")
# hf_dataset = hf_dataset.map(prepare_dataset, remove_columns=hf_dataset.column_names, num_proc=1)
# print(f"Processed {len(hf_dataset)} examples")

# # ============================================


In [5]:
from torch.utils.data import random_split
print("\n" + "=" * 70)
print("PART 2: PREPARE DATA FOR WHISPER")
print("=" * 70)
import torchaudio
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

class BurushaskiDataset(torch.utils.data.Dataset):
    def __init__(self, manifest, processor):
        self.manifest = manifest
        self.processor = processor

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        item = self.manifest[idx]
        waveform, sr = torchaudio.load(item['audio_path'])  # not librosa
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)
        audio = waveform.squeeze().numpy()
        inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt")
        labels = self.processor.tokenizer(item['text'], return_tensors="pt").input_ids
        return {
            "input_features": inputs.input_features.squeeze(0),
            "labels": labels.squeeze(0)
        }

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.use_cache = False
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="translate")
model.generation_config.suppress_tokens = []

full_dataset = BurushaskiDataset(train_manifest, processor)
val_size = int(0.1 * len(full_dataset))
train_data, eval_data = random_split(full_dataset, [len(full_dataset)-val_size, val_size],
                                      generator=torch.Generator().manual_seed(42))
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")


PART 2: PREPARE DATA FOR WHISPER


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Train: 10779, Eval: 1197


In [6]:
# # ============================================
# # PART 2: LOAD PROCESSOR + MODEL
# # ============================================

# print("\n" + "=" * 70)
# print("PART 2: PREPARE DATA FOR WHISPER")
# print("=" * 70)

# processor = WhisperProcessor.from_pretrained("openai/whisper-small")
# print("Processor loaded")

# model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
# # model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="translate")
# # model.config.suppress_tokens = []
# # model.config.use_cache = False
# model.config.use_cache = False
# model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="translate")
# model.generation_config.suppress_tokens = []
# print("Model loaded")

# train_dataset = BurushaskiDataset(train_manifest, processor)
# print(f"Dataset ready: {len(train_dataset)} examples (lazy loading)")

In [7]:
 # TRAIN WHISPER
# ============================================

print("\n" + "=" * 70)
print("PART 3: TRAIN WHISPER")
print("=" * 70)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.config.use_cache = False
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# # Split for validation
# train_test_split = hf_dataset.train_test_split(test_size=0.1, seed=42)
# train_data = train_test_split["train"]
# eval_data = train_test_split["test"]

print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper_s_checkpoint",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=200,
    num_train_epochs=5,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    per_device_eval_batch_size=16,
    predict_with_generate=False,
    generation_max_length=225,
    logging_steps=25,
    report_to="none",
    push_to_hub=False,
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator,
    processing_class=processor,
)

print("\nStarting training...")
# trainer.train(resume_from_checkpoint=True)
import os
checkpoint = None
if os.path.isdir("./whisper_s_checkpoint") and any("checkpoint" in f for f in os.listdir("./whisper_s_checkpoint")):
    checkpoint = "./whisper_s_checkpoint"

trainer.train(resume_from_checkpoint=checkpoint)

model.save_pretrained("./whisper_burushaski_small_final")
processor.save_pretrained("./whisper_burushaski_small_final")
print("\nModel saved")

# ============================================



PART 3: TRAIN WHISPER
Train: 10779, Eval: 1197

Starting training...


Step,Training Loss,Validation Loss
200,2.752665,1.400511
400,0.931141,0.795125
600,0.522244,0.753736
800,0.356335,0.766982


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved


In [8]:
# model.save_pretrained("./test_save")
# print("Save test passed — safe to train")

In [9]:
# # ============================================
# # PART 3: TRAIN WHISPER
# # ============================================

# print("\n" + "=" * 70)
# print("PART 3: TRAIN WHISPER")
# print("=" * 70)

# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# model.config.use_cache = False
# model.generation_config.forced_decoder_ids = None
# model.generation_config.suppress_tokens = []

# @dataclass
# class DataCollatorSpeechSeq2SeqWithPadding:
#     processor: Any

#     def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
#         input_features = [{"input_features": f["input_features"]} for f in features]
#         batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
#         label_features = [{"input_ids": f["labels"]} for f in features]
#         labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
#         labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
#         if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
#             labels = labels[:, 1:]
#         batch["labels"] = labels
#         return batch

# data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# from torch.utils.data import random_split
# val_size = int(0.1 * len(train_dataset))
# train_size = len(train_dataset) - val_size
# train_data, eval_data = random_split(
#     train_dataset, [train_size, val_size],
#     generator=torch.Generator().manual_seed(42)
# )
# print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

# training_args = Seq2SeqTrainingArguments(
#     output_dir="./whisper_burushaski_small_augmented",
#     per_device_train_batch_size=16,   # T4 can handle 16 for Small
#     gradient_accumulation_steps=2,
#     learning_rate=1e-5,
#     warmup_steps=200,
#     num_train_epochs=10,
#     gradient_checkpointing=True,
#     fp16=True,
#     eval_strategy="steps",
#     eval_steps=500,
#     save_strategy="steps",
#     save_steps=500,
#     save_total_limit=2,
#     load_best_model_at_end=True,
#     metric_for_best_model="loss",
#     greater_is_better=False,
#     per_device_eval_batch_size=16,
#     predict_with_generate=True,
#     generation_max_length=225,
#     logging_steps=25,
#     report_to="none",
#     push_to_hub=False,
#     remove_unused_columns=False,
#     dataloader_num_workers=2,
# )

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = model.to(device)

# trainer = Seq2SeqTrainer(
#     args=training_args,
#     model=model,
#     train_dataset=train_data,
#     eval_dataset=eval_data,
#     data_collator=data_collator,
#     processing_class=processor,
# )

# print("\nStarting training...")
# trainer.train()

# model.save_pretrained("./whisper_burushaski_small_augmented_final")
# processor.save_pretrained("./whisper_burushaski_small_augmented_final")
# print("\nModel saved")

In [10]:
 # LOAD TEST DATA
# ============================================

print("\n" + "=" * 70)
print("PART 4: LOAD TEST DATA")
print("=" * 70)

# UPDATE THESE PATHS
TEST_AUDIO_DIR = "/kaggle/input/datasets/azkaanasir/15kdatasetapp-mozilla/15ktranlsateddata/3ktest_dataset/test/AUDIO"
TEST_TXT_DIR = "/kaggle/input/datasets/azkaanasir/15kdatasetapp-mozilla/15ktranlsateddata/3ktest_dataset/test/TXT"

print(f"\nTest data paths:")
print(f"  Audio: {TEST_AUDIO_DIR}")
print(f"  Text:  {TEST_TXT_DIR}")

test_audio_files = sorted([f for f in os.listdir(TEST_AUDIO_DIR) if f.endswith('.wav')])
test_txt_files = sorted([f for f in os.listdir(TEST_TXT_DIR) if f.endswith('.txt')])

# Match files
text_lookup = {os.path.splitext(f)[0]: f for f in test_txt_files}
test_pairs = []

for audio_file in test_audio_files:
    stem = os.path.splitext(audio_file)[0]
    if stem in text_lookup:
        test_pairs.append((audio_file, text_lookup[stem]))

print(f"\nMatched test pairs: {len(test_pairs)}")

# ============================================



PART 4: LOAD TEST DATA

Test data paths:
  Audio: /kaggle/input/datasets/azkaanasir/15kdatasetapp-mozilla/15ktranlsateddata/3ktest_dataset/test/AUDIO
  Text:  /kaggle/input/datasets/azkaanasir/15kdatasetapp-mozilla/15ktranlsateddata/3ktest_dataset/test/TXT

Matched test pairs: 2994


In [11]:
#  #  EVALUATION
# # ============================================

# print("\n" + "=" * 70)
# print("PART 5: COMPREHENSIVE EVALUATION")
# print("=" * 70)

# # Reload model for inference
# model = WhisperForConditionalGeneration.from_pretrained("./whisper_burushaski_small_augmented_final")
# processor = WhisperProcessor.from_pretrained("./whisper_burushaski_small_augmented_final")
# model = model.to(device)
# model.eval()

# # Run inference on test set
# hypotheses = []
# references = []
# filenames = []
# failed = []

# print("\nRunning inference on test set...")
# for audio_file, text_file in tqdm(test_pairs, desc="Evaluating"):
#     audio_path = os.path.join(TEST_AUDIO_DIR, audio_file)
#     text_path = os.path.join(TEST_TXT_DIR, text_file)
    
#     try:
#         with open(text_path, 'r', encoding='utf-8') as f:
#             reference = f.read().strip()
        
#         audio, _ = librosa.load(audio_path, sr=16000)
        
#         inputs = processor(audio, sampling_rate=16000, return_tensors='pt').input_features.to(device)
        
#         with torch.no_grad():
#             predicted_ids = model.generate(inputs, max_length=225)
        
#         hypothesis = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
        
#         hypotheses.append(hypothesis)
#         references.append(reference)
#         filenames.append(audio_file)
    
#     except Exception as e:
#         print(f"  Failed: {audio_file} — {e}")
#         failed.append(audio_file)

# print(f"\nProcessed: {len(hypotheses)} | Failed: {len(failed)}")

# # ============================================
# # CALCULATE METRICS (WITH PREPROCESSING)
# # ============================================

# import string

# print("\n" + "=" * 70)
# print("CALCULATING METRICS")
# print("=" * 70)

# # Preprocessing function
# def normalize_text(text):
#     """Lowercase, remove punctuation, strip spaces"""
#     # Convert to lowercase
#     text = text.lower()
#     # Remove punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     # Strip leading/trailing spaces and collapse multiple spaces
#     text = ' '.join(text.split())
#     return text

# # Normalize all predictions and references
# print("\nNormalizing text (lowercase, remove punctuation)...")
# hypotheses_normalized = [normalize_text(h) for h in hypotheses]
# references_normalized = [normalize_text(r) for r in references]

# print(f"Example normalization:")
# print(f"  Original:   '{hypotheses[0]}'")
# print(f"  Normalized: '{hypotheses_normalized[0]}'")

# # 1. BLEU (from evaluate)
# print("\n[1/6] Computing BLEU...")
# bleu_metric = evaluate.load("sacrebleu")
# bleu_result = bleu_metric.compute(
#     predictions=hypotheses_normalized, 
#     references=[[r] for r in references_normalized]
# )
# bleu_score = bleu_result['score']

# # 2. BLEU (corpus-level with sacrebleu)
# bleu_corpus = BLEU().corpus_score(hypotheses_normalized, [references_normalized])

# # 3. chrF++
# print("[2/6] Computing chrF++...")
# chrf_result = CHRF(word_order=2).corpus_score(hypotheses_normalized, [references_normalized])

# # 4. WER & CER
# print("[3/6] Computing WER & CER...")
# wer = jiwer.wer(references_normalized, hypotheses_normalized)
# cer = jiwer.cer(references_normalized, hypotheses_normalized)

# # 5. Word Overlap
# print("[4/6] Computing Word Overlap...")
# word_accuracies = []
# for pred, ref in zip(hypotheses_normalized, references_normalized):
#     pred_words = set(pred.split())  # Already lowercase, no punctuation
#     ref_words = set(ref.split())
#     if len(ref_words) > 0:
#         overlap = len(pred_words & ref_words) / len(ref_words)
#         word_accuracies.append(overlap)

# word_overlap = np.mean(word_accuracies) * 100

# # 6. BERTScore (use original text, not normalized)
# print("[5/6] Computing BERTScore (takes ~1 min)...")
# valid_pairs = [(h, r) for h, r in zip(hypotheses, references) if h.strip() and r.strip()]
# hyp_clean = [p[0] for p in valid_pairs]
# ref_clean = [p[1] for p in valid_pairs]

# try:
#     P, R, F1 = bert_score(
#         hyp_clean, ref_clean,
#         model_type='bert-base-multilingual-cased',
#         verbose=False
#     )
#     bertscore_f1 = F1.mean().item()
# except Exception as e:
#     print(f"BERTScore failed: {e}")
#     bertscore_f1 = 0.0

# # 7. Length statistics (use normalized)
# print("[6/6] Computing length statistics...")
# pred_lengths = [len(p.split()) for p in hypotheses_normalized]
# ref_lengths = [len(r.split()) for r in references_normalized]


# # ============================================
# # PRINT RESULTS
# # ============================================

# print("\n" + "=" * 70)
# print("EVALUATION RESULTS SUMMARY")
# print("=" * 70)

# print(f"\nDataset:")
# print(f"  Test samples evaluated: {len(hypotheses)}")
# print(f"  Failed: {len(failed)}")

# print(f"\nTranslation Quality Metrics:")
# print(f"  BLEU Score (0-100):     {bleu_score:.2f}")
# print(f"  chrF++ (0-100):         {chrf_result.score:.2f}")
# print(f"  BERTScore F1 (0-1):     {bertscore_f1:.4f}")

# print(f"\nWord-Level Accuracy:")
# print(f"  WER (lower=better):     {wer*100:.2f}%")
# print(f"  Word Accuracy:          {(1-wer)*100:.2f}%")
# print(f"  Word Overlap:           {word_overlap:.2f}%")

# print(f"\nCharacter-Level:")
# print(f"  CER (lower=better):     {cer*100:.2f}%")

# print(f"\nLength Statistics:")
# print(f"  Avg prediction length:  {np.mean(pred_lengths):.1f} words")
# print(f"  Avg reference length:   {np.mean(ref_lengths):.1f} words")
# print(f"  Length ratio:           {np.mean(pred_lengths)/np.mean(ref_lengths):.2f}")

# # Quality interpretation
# if bleu_score < 10:
#     quality = "Poor"
# elif bleu_score < 20:
#     quality = "Fair"
# elif bleu_score < 30:
#     quality = "Good"
# else:
#     quality = "Excellent"

# print(f"\nOverall Quality: {quality}")


# # ============================================
# # SAMPLE PREDICTIONS (Show both versions)
# # ============================================

# print("\n" + "=" * 70)
# print("SAMPLE PREDICTIONS (First 10)")
# print("=" * 70)

# for i in range(min(10, len(hypotheses))):
#     sample_wer = jiwer.wer(references_normalized[i], hypotheses_normalized[i]) * 100
#     print(f"\n[{i+1}] {filenames[i]}")
#     print(f"  REF (original):    {references[i]}")
#     print(f"  HYP (original):    {hypotheses[i]}")
#     print(f"  REF (normalized):  {references_normalized[i]}")
#     print(f"  HYP (normalized):  {hypotheses_normalized[i]}")
#     print(f"  WER: {sample_wer:.1f}%")
    
# # ============================================
# # SAVE RESULTS
# # ============================================

# print("\n" + "=" * 70)
# print("SAVING RESULTS")
# print("=" * 70)

# # Per-sample results CSV
# results_df = pd.DataFrame({
#     'filename': filenames,
#     'reference': references,
#     'hypothesis': hypotheses,
#     'sample_wer_%': [round(jiwer.wer(r, h) * 100, 2) for r, h in zip(references, hypotheses)],
#     'sample_cer_%': [round(jiwer.cer(r, h) * 100, 2) for r, h in zip(references, hypotheses)],
# })
# results_df.to_csv('evaluation_results_detailed.csv', index=False)
# print("Saved: evaluation_results_detailed.csv")

# # Summary metrics CSV
# summary_df = pd.DataFrame([{
#     'Total_Samples': len(hypotheses),
#     'BLEU': round(bleu_score, 2),
#     'chrF++': round(chrf_result.score, 2),
#     'BERTScore_F1': round(bertscore_f1, 4),
#     'WER_%': round(wer * 100, 2),
#     'CER_%': round(cer * 100, 2),
#     'Word_Accuracy_%': round((1-wer) * 100, 2),
#     'Word_Overlap_%': round(word_overlap, 2),
#     'Quality': quality
# }])
# summary_df.to_csv('metrics_summary.csv', index=False)
# print("Saved: metrics_summary.csv")

# # Complete results pickle
# complete_results = {
#     'metrics': {
#         'bleu': bleu_score,
#         'chrf': chrf_result.score,
#         'bertscore_f1': bertscore_f1,
#         'wer': wer * 100,
#         'cer': cer * 100,
#         'word_overlap': word_overlap,
#         'quality': quality
#     },
#     'predictions': hypotheses,
#     'references': references,
#     'filenames': filenames
# }

# with open('complete_evaluation_results.pkl', 'wb') as f:
#     pickle.dump(complete_results, f)
# print("Saved: complete_evaluation_results.pkl")

# # Text report
# # with open('evaluation_report.txt', 'w', encoding='utf-8') as f:
# #     f.write("=" * 70 + "\n")
# #     f.write("WHISPER BURUSHASKI → ENGLISH EVALUATION REPORT\n")
# #     f.write("=" * 70 + "\n\n")
    
# #     f.write(f"Test samples: {len(hypotheses)}\n")
# #     f.write(f"BLEU: {bleu_score:.2f}\n")
# #     f.write(f"chrF++: {chrf_result.score:.2f}\n")
# #     f.write(f"BERTScore F1: {bertscore_f1:.4f}\n")
# #     f.write(f"WER: {wer*100:.2f}%\n")
# #     f.write(f"Word Accuracy: {(1-wer)*100:.2f}%\n")
# #     f.write(f"Quality: {quality}\n\n")
    
# #     f.write("=" * 70 + "\n")
# #     f.write("SAMPLE PREDICTIONS\n")
# #     f.write("=" * 70 + "\n\n")
    
# #     for i in range(len(hypotheses)):
# #         f.write(f"[{i+1}] {filenames[i]}\n")
# #         f.write(f"REF: {references[i]}\n")
# #         f.write(f"HYP: {hypotheses[i]}\n\n")

# # print("Saved: evaluation_report.txt")

# # ============================================
# # ZIP EVERYTHING
# # ============================================

# print("\n" + "=" * 70)
# print("CREATING FINAL ZIP")
# print("=" * 70)

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# final_zip = f"whisper_complete_{timestamp}.zip"

# with zipfile.ZipFile(final_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
#     # Add model
#     for root, dirs, files in os.walk('./whisper_burushaski_medium_final'):
#         for file in files:
#             zipf.write(os.path.join(root, file))
    
#     # Add results
#     for f in ['evaluation_results_detailed.csv', 'metrics_summary.csv', 
#               'complete_evaluation_results.pkl', 'evaluation_report.txt']:
#         if os.path.exists(f):
#             zipf.write(f, f"results/{f}")

# zip_size = os.path.getsize(final_zip) / (1024 * 1024)
# print(f"\nCreated: {final_zip} ({zip_size:.2f} MB)")

# print("\n" + "=" * 70)
# print("PIPELINE COMPLETE")
# print("=" * 70)
# print(f"\nFinal BLEU: {bleu_score:.2f}")
# print(f"Word Accuracy: {(1-wer)*100:.2f}%")
# print(f"Quality: {quality}")
# print("=" * 70)


In [12]:
 #  EVALUATION
# ============================================

print("\n" + "=" * 70)
print("PART 5: COMPREHENSIVE EVALUATION")
print("=" * 70)

# Reload model for inference
# model = WhisperForConditionalGeneration.from_pretrained("./whisper_burushaski_medium_final")
# processor = WhisperProcessor.from_pretrained("./whisper_burushaski_medium_final")
# model = WhisperForConditionalGeneration.from_pretrained("./whisper_burushaski_large_final")
# processor = WhisperProcessor.from_pretrained("./whisper_burushaski_large_final")
model = WhisperForConditionalGeneration.from_pretrained("./whisper_burushaski_small_final")
processor = WhisperProcessor.from_pretrained("./whisper_burushaski_small_final")
model = model.to(device)
model.eval()

# Run inference on test set
hypotheses = []
references = []
filenames = []
failed = []

print("\nRunning inference on test set...")
for audio_file, text_file in tqdm(test_pairs, desc="Evaluating"):
    audio_path = os.path.join(TEST_AUDIO_DIR, audio_file)
    text_path = os.path.join(TEST_TXT_DIR, text_file)

    try:
        with open(text_path, 'r', encoding='utf-8') as f:
            reference = f.read().strip()

        audio, _ = librosa.load(audio_path, sr=16000)

        inputs = processor(audio, sampling_rate=16000, return_tensors='pt').input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(inputs, max_length=225)

        hypothesis = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()

        hypotheses.append(hypothesis)
        references.append(reference)
        filenames.append(audio_file)

    except Exception as e:
        print(f"  Failed: {audio_file} — {e}")
        failed.append(audio_file)

print(f"\nProcessed: {len(hypotheses)} | Failed: {len(failed)}")

# ============================================
# CALCULATE METRICS (WITH PREPROCESSING)
# ============================================

import string

print("\n" + "=" * 70)
print("CALCULATING METRICS")
print("=" * 70)

# Preprocessing function
def normalize_text(text):
    """Lowercase, remove punctuation, strip spaces"""
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Strip leading/trailing spaces and collapse multiple spaces
    text = ' '.join(text.split())
    return text

# Normalize all predictions and references
print("\nNormalizing text (lowercase, remove punctuation)...")
hypotheses_normalized = [normalize_text(h) for h in hypotheses]
references_normalized = [normalize_text(r) for r in references]

print(f"Example normalization:")
print(f"  Original:   '{hypotheses[0]}'")
print(f"  Normalized: '{hypotheses_normalized[0]}'")

# 1. BLEU (from evaluate)
print("\n[1/6] Computing BLEU...")
bleu_metric = evaluate.load("sacrebleu")
bleu_result = bleu_metric.compute(
    predictions=hypotheses_normalized,
    references=[[r] for r in references_normalized]
)
bleu_score = bleu_result['score']

# 2. BLEU (corpus-level with sacrebleu)
bleu_corpus = BLEU().corpus_score(hypotheses_normalized, [references_normalized])

# 3. chrF++
print("[2/6] Computing chrF++...")
chrf_result = CHRF(word_order=2).corpus_score(hypotheses_normalized, [references_normalized])

# 4. WER & CER
print("[3/6] Computing WER & CER...")
wer = jiwer.wer(references_normalized, hypotheses_normalized)
cer = jiwer.cer(references_normalized, hypotheses_normalized)

# 5. Word Overlap
print("[4/6] Computing Word Overlap...")
word_accuracies = []
for pred, ref in zip(hypotheses_normalized, references_normalized):
    pred_words = set(pred.split())  # Already lowercase, no punctuation
    ref_words = set(ref.split())
    if len(ref_words) > 0:
        overlap = len(pred_words & ref_words) / len(ref_words)
        word_accuracies.append(overlap)

word_overlap = np.mean(word_accuracies) * 100

# 6. BERTScore (use original text, not normalized)
print("[5/6] Computing BERTScore (takes ~1 min)...")
valid_pairs = [(h, r) for h, r in zip(hypotheses, references) if h.strip() and r.strip()]
hyp_clean = [p[0] for p in valid_pairs]
ref_clean = [p[1] for p in valid_pairs]

try:
    P, R, F1 = bert_score(
        hyp_clean, ref_clean,
        model_type='bert-base-multilingual-cased',
        verbose=False
    )
    bertscore_f1 = F1.mean().item()
except Exception as e:
    print(f"BERTScore failed: {e}")
    bertscore_f1 = 0.0

# 7. Length statistics (use normalized)
print("[6/6] Computing length statistics...")
pred_lengths = [len(p.split()) for p in hypotheses_normalized]
ref_lengths = [len(r.split()) for r in references_normalized]


# ============================================
# PRINT RESULTS
# ============================================

print("\n" + "=" * 70)
print("EVALUATION RESULTS SUMMARY")
print("=" * 70)

print(f"\nDataset:")
print(f"  Test samples evaluated: {len(hypotheses)}")
print(f"  Failed: {len(failed)}")

print(f"\nTranslation Quality Metrics:")
print(f"  BLEU Score (0-100):     {bleu_score:.2f}")
print(f"  chrF++ (0-100):         {chrf_result.score:.2f}")
print(f"  BERTScore F1 (0-1):     {bertscore_f1:.4f}")

print(f"\nWord-Level Accuracy:")
print(f"  WER (lower=better):     {wer*100:.2f}%")
print(f"  Word Accuracy:          {(1-wer)*100:.2f}%")
print(f"  Word Overlap:           {word_overlap:.2f}%")

print(f"\nCharacter-Level:")
print(f"  CER (lower=better):     {cer*100:.2f}%")

print(f"\nLength Statistics:")
print(f"  Avg prediction length:  {np.mean(pred_lengths):.1f} words")
print(f"  Avg reference length:   {np.mean(ref_lengths):.1f} words")
print(f"  Length ratio:           {np.mean(pred_lengths)/np.mean(ref_lengths):.2f}")

# Quality interpretation
if bleu_score < 10:
    quality = "Poor"
elif bleu_score < 20:
    quality = "Fair"
elif bleu_score < 30:
    quality = "Good"
else:
    quality = "Excellent"

print(f"\nOverall Quality: {quality}")


# ============================================
# SAMPLE PREDICTIONS (Show both versions)
# ============================================

print("\n" + "=" * 70)
print("SAMPLE PREDICTIONS (First 10)")
print("=" * 70)

for i in range(min(10, len(hypotheses))):
    sample_wer = jiwer.wer(references_normalized[i], hypotheses_normalized[i]) * 100
    print(f"\n[{i+1}] {filenames[i]}")
    print(f"  REF (original):    {references[i]}")
    print(f"  HYP (original):    {hypotheses[i]}")
    print(f"  REF (normalized):  {references_normalized[i]}")
    print(f"  HYP (normalized):  {hypotheses_normalized[i]}")
    print(f"  WER: {sample_wer:.1f}%")

# ============================================
# SAVE RESULTS
# ============================================

print("\n" + "=" * 70)
print("SAVING RESULTS")
print("=" * 70)

# Per-sample results CSV
results_df = pd.DataFrame({
    'filename': filenames,
    'reference': references,
    'hypothesis': hypotheses,
    'references_normalized' : references_normalized,
    'hypothesis_normalized' : hypotheses_normalized,
    'sample_wer_%': [round(jiwer.wer(r, h) * 100, 2) for r, h in zip(references_normalized, hypotheses_normalized)],
    'sample_cer_%': [round(jiwer.cer(r, h) * 100, 2) for r, h in zip(references_normalized, hypotheses_normalized)],
})
results_df.to_csv('evaluation_results_detailed.csv', index=False)
print("Saved: evaluation_results_detailed.csv")

# Summary metrics CSV
summary_df = pd.DataFrame([{
    'Total_Samples': len(hypotheses),
    'BLEU': round(bleu_score, 2),
    'chrF++': round(chrf_result.score, 2),
    'BERTScore_F1': round(bertscore_f1, 4),
    'WER_%': round(wer * 100, 2),
    'CER_%': round(cer * 100, 2),
    'Word_Accuracy_%': round((1-wer) * 100, 2),
    'Word_Overlap_%': round(word_overlap, 2),
    'Quality': quality
}])
summary_df.to_csv('metrics_summary.csv', index=False)
print("Saved: metrics_summary.csv")

# Complete results pickle
complete_results = {
    'metrics': {
        'bleu': bleu_score,
        'chrf': chrf_result.score,
        'bertscore_f1': bertscore_f1,
        'wer': wer * 100,
        'cer': cer * 100,
        'word_overlap': word_overlap,
        'quality': quality
    },
    'predictions': hypotheses,
    'references': references,
    'filenames': filenames
}

with open('complete_evaluation_results.pkl', 'wb') as f:
    pickle.dump(complete_results, f)
print("Saved: complete_evaluation_results.pkl")

# Text report
# with open('evaluation_report.txt', 'w', encoding='utf-8') as f:
#     f.write("=" * 70 + "\n")
#     f.write("WHISPER BURUSHASKI → ENGLISH EVALUATION REPORT\n")
#     f.write("=" * 70 + "\n\n")

#     f.write(f"Test samples: {len(hypotheses)}\n")
#     f.write(f"BLEU: {bleu_score:.2f}\n")
#     f.write(f"chrF++: {chrf_result.score:.2f}\n")
#     f.write(f"BERTScore F1: {bertscore_f1:.4f}\n")
#     f.write(f"WER: {wer*100:.2f}%\n")
#     f.write(f"Word Accuracy: {(1-wer)*100:.2f}%\n")
#     f.write(f"Quality: {quality}\n\n")

#     f.write("=" * 70 + "\n")
#     f.write("SAMPLE PREDICTIONS\n")
#     f.write("=" * 70 + "\n\n")

#     for i in range(len(hypotheses)):
#         f.write(f"[{i+1}] {filenames[i]}\n")
#         f.write(f"REF: {references[i]}\n")
#         f.write(f"HYP: {hypotheses[i]}\n\n")

# print("Saved: evaluation_report.txt")

# ============================================
# ZIP EVERYTHING
# ============================================

print("\n" + "=" * 70)
print("CREATING FINAL ZIP")
print("=" * 70)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
final_zip = f"whisper_complete_{timestamp}.zip"

with zipfile.ZipFile(final_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add model
    for root, dirs, files in os.walk('./whisper_burushaski_large_final'):
        for file in files:
            zipf.write(os.path.join(root, file))

    # Add results
    for f in ['evaluation_results_detailed.csv', 'metrics_summary.csv',
              'complete_evaluation_results.pkl', 'evaluation_report.txt']:
        if os.path.exists(f):
            zipf.write(f, f"results/{f}")

zip_size = os.path.getsize(final_zip) / (1024 * 1024)
print(f"\nCreated: {final_zip} ({zip_size:.2f} MB)")

print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)
print(f"\nFinal BLEU: {bleu_score:.2f}")
print(f"Word Accuracy: {(1-wer)*100:.2f}%")
print(f"Quality: {quality}")
print("=" * 70)



PART 5: COMPREHENSIVE EVALUATION


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


Running inference on test set...


Evaluating:   0%|          | 0/2994 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBegin


Processed: 2994 | Failed: 0

CALCULATING METRICS

Normalizing text (lowercase, remove punctuation)...
Example normalization:
  Original:   'These are potatoes in the pot.'
  Normalized: 'these are potatoes in the pot'

[1/6] Computing BLEU...


...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[6/6] Computing length statistics...

EVALUATION RESULTS SUMMARY

Dataset:
  Test samples evaluated: 2994
  Failed: 0

Translation Quality Metrics:
  BLEU Score (0-100):     74.10
  chrF++ (0-100):         78.72
  BERTScore F1 (0-1):     0.9388

Word-Level Accuracy:
  WER (lower=better):     24.33%
  Word Accuracy:          75.67%
  Word Overlap:           82.54%

Character-Level:
  CER (lower=better):     19.29%

Length Statistics:
  Avg prediction length:  5.3 words
  Avg reference length:   5.3 words
  Length ratio:           1.00

Overall Quality: Excellent

SAMPLE PREDICTIONS (First 10)

[1] 41893803.wav
  REF (original):    There are potatoes in this sac
  HYP (original):    These are potatoes in the pot.
  REF (normalized):  there are potatoes in this sac
  HYP (normalized):  these are potatoes in the pot
  WER: 50.0%

[2] 41894051.wav
  REF (original):    I will not go to the party
  HYP (original):    I will not go to the party.
  REF (normalized):  i will not go to the party
